In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv(r'C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\earningsExtractionSummary.csv')
pricesDF = pd.read_csv(r'C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\companyPrices.csv')
directionDF = pd.read_csv(r'C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\direction.csv')

In [3]:
pricesDF['PriceChangeD+1']  = (pricesDF['CloseDay+1_Price'] - pricesDF['CloseDay_Price']) / pricesDF['CloseDay_Price']
pricesDF['PriceChangeD+7']  = (pricesDF['CloseDay+7_Price'] - pricesDF['CloseDay_Price']) / pricesDF['CloseDay_Price']
pricesDF['PriceChangeD+28'] = (pricesDF['CloseDay+28_Price'] - pricesDF['CloseDay_Price']) / pricesDF['CloseDay_Price']
pricesDF['PriceChangeD+1_%']  = pricesDF['PriceChangeD+1']  * 100
pricesDF['PriceChangeD+7_%']  = pricesDF['PriceChangeD+7']  * 100
pricesDF['PriceChangeD+28_%'] = pricesDF['PriceChangeD+28'] * 100
pricesDF = pricesDF.round({
    'PriceChangeD+1': 4,
    'PriceChangeD+7': 4,
    'PriceChangeD+28': 4,
	'PriceChangeD+1_%': 2,
    'PriceChangeD+7_%': 2,
    'PriceChangeD+28_%': 2
})

In [4]:
directionDF = directionDF.merge(
    pricesDF[['Ticker', 'Quarter', 'Year', 
              'PriceChangeD+1_%', 'PriceChangeD+7_%', 'PriceChangeD+28_%']],
    on=['Ticker','Quarter','Year'],
    how='left'
)

In [5]:
sentiment_counts = (
    df
    .groupby(['Ticker', 'Quarter', 'Year', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

sentiment_counts['dominant_sentiment'] = sentiment_counts[['positive', 'negative', 'neutral']].idxmax(axis=1)
sentiment_counts['sentiment_diff'] = sentiment_counts['positive'] - sentiment_counts['negative']

cols_to_merge = ['Ticker', 'Quarter', 'Year', 'PriceChangeD+1_%', 'PriceChangeD+7_%', 'PriceChangeD+28_%']
sentiment_counts = sentiment_counts.merge(pricesDF[cols_to_merge].drop_duplicates(), on=['Ticker','Quarter','Year'], how='left')

sentiment_counts['recommendation'] = np.where(
    sentiment_counts['dominant_sentiment'] == 'negative',
    'SELL',
    'BUY'
)

print(sentiment_counts.head())

sentiment_counts.to_csv(r"C:\Users\phata\OneDrive - George Mason University - O365 Production\AIT626\05. Projects\Final Project\Earnings2Insights\sentiment_recommendations.csv", index=False)

  Ticker  Quarter  Year  negative  neutral  positive dominant_sentiment  \
0    ABM        3  2021         0       24         0            neutral   
1    AME        1  2021        10       13         5            neutral   
2  ATLKY        2  2016         4        1         0           negative   
3  BAESY        4  2019        11       22         1            neutral   
4    CFR        3  2019         0        1         0            neutral   

   sentiment_diff  PriceChangeD+1_%  PriceChangeD+7_%  PriceChangeD+28_%  \
0               0              0.00             -2.42               0.24   
1              -5              0.62              0.08              -0.19   
2              -4              0.00              0.58               4.35   
3             -10              2.22             -4.88             -32.42   
4               0              1.04              4.74               4.06   

  recommendation  
0            BUY  
1            BUY  
2           SELL  
3            BUY

In [6]:
mergedDF = sentiment_counts.merge(
    directionDF[['Ticker', 'Quarter', 'Year', 'direction']],
    on=['Ticker', 'Quarter', 'Year'],
    how='left'
)

In [7]:
price_cols = ['PriceChangeD+1_%', 'PriceChangeD+7_%', 'PriceChangeD+28_%']

accuracy_results = {}

for col in price_cols:
    correct_mask = (
        ((sentiment_counts['recommendation'] == 'BUY')  & (sentiment_counts[col] >= 0)) |
        ((sentiment_counts['recommendation'] == 'SELL') & (sentiment_counts[col] < 0))
    )

    total = sentiment_counts[col].notna().sum()
    correct = correct_mask.sum()
    accuracy = (correct / total) * 100 if total > 0 else 0

    accuracy_results[col] = round(accuracy, 2)

# Display accuracy summary
accuracy_df = pd.DataFrame.from_dict(accuracy_results, orient='index', columns=['Accuracy_%'])
print(accuracy_df)


                   Accuracy_%
PriceChangeD+1_%        55.07
PriceChangeD+7_%        55.07
PriceChangeD+28_%       66.67


In [8]:
price_cols = ['PriceChangeD+1_%', 'PriceChangeD+7_%', 'PriceChangeD+28_%']
direction_accuracy = {}

for col in price_cols:
    correct_mask = (
        ((directionDF['direction'] == 'UP')   & (directionDF[col] >= 0)) |
        ((directionDF['direction'] == 'DOWN') & (directionDF[col] < 0))
    )

    total = directionDF[col].notna().sum()
    correct = correct_mask.sum()
    accuracy = (correct / total) * 100 if total > 0 else 0

    direction_accuracy[col] = round(accuracy, 2)

direction_accuracy_df = pd.DataFrame.from_dict(
    direction_accuracy, orient='index', columns=['Accuracy_%']
)

print("\nAccuracy using direction only:")
print(direction_accuracy_df)



Accuracy using direction only:
                   Accuracy_%
PriceChangeD+1_%        45.31
PriceChangeD+7_%        46.88
PriceChangeD+28_%       57.81


In [9]:
mergedDF['direction'] = mergedDF['direction'].astype('category')
mergedDF['recommendation'] = mergedDF['recommendation'].astype('category')

mergedDF['Target_D1']  = (mergedDF['PriceChangeD+1_%']  >= 0).astype(int)
mergedDF['Target_D7']  = (mergedDF['PriceChangeD+7_%']  >= 0).astype(int)
mergedDF['Target_D28'] = (mergedDF['PriceChangeD+28_%'] >= 0).astype(int)

encoder = LabelEncoder()

mergedDF['direction_enc'] = encoder.fit_transform(mergedDF['direction'])
mergedDF['recommendation_enc'] = encoder.fit_transform(mergedDF['recommendation'])

In [10]:
rf  = RandomForestClassifier(n_estimators=500, random_state=42)
gb  = GradientBoostingClassifier(n_estimators=300, random_state=42)
xgb = XGBClassifier(n_estimators=300, random_state=42, eval_metric='logloss')

ensemble = VotingClassifier(
    estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)],
    voting='soft'
)

In [13]:
def train_ensemble_for_horizon(df, target_col):

    features = ['direction_enc', 'recommendation_enc']

    X = df[features]
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42
    )

    rf  = RandomForestClassifier(n_estimators=500, random_state=42)
    gb  = GradientBoostingClassifier(n_estimators=300, random_state=42)
    xgb = XGBClassifier(n_estimators=300, random_state=42, eval_metric='logloss')

    ensemble = VotingClassifier(
        estimators=[('rf', rf), ('gb', gb), ('xgb', xgb)],
        voting='soft'
    )

    ensemble.fit(X_train, y_train)

    preds = ensemble.predict(X_test)
    acc = accuracy_score(y_test, preds)

    print(f"\n=== Ensemble Model for {target_col} ===")
    print(f"Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, preds))

    return ensemble


In [14]:
ens_D1  = train_ensemble_for_horizon(mergedDF, 'Target_D1')
ens_D7  = train_ensemble_for_horizon(mergedDF, 'Target_D7')
ens_D28 = train_ensemble_for_horizon(mergedDF, 'Target_D28')


=== Ensemble Model for Target_D1 ===
Accuracy: 42.86%
              precision    recall  f1-score   support

           0       0.45      0.71      0.56         7
           1       0.33      0.14      0.20         7

    accuracy                           0.43        14
   macro avg       0.39      0.43      0.38        14
weighted avg       0.39      0.43      0.38        14


=== Ensemble Model for Target_D7 ===
Accuracy: 28.57%
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         8
           1       0.33      0.67      0.44         6

    accuracy                           0.29        14
   macro avg       0.17      0.33      0.22        14
weighted avg       0.14      0.29      0.19        14


=== Ensemble Model for Target_D28 ===
Accuracy: 71.43%
              precision    recall  f1-score   support

           0       1.00      0.20      0.33         5
           1       0.69      1.00      0.82         9

    accuracy    